# Family Reversal Analysis: Qwen2.5 / Qwen3.5 / Gemma-3

Three families, all dense (active params = total params).  
Question: at what K/N does each model reverse, and what determines that boundary?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ── model catalogue ──────────────────────────────────────────────────────────
MODELS = {
    # model_id:  (params_B, family, short_label, n_layers, n_heads, d_model)
    'Qwen2.5-0.5B-Instruct': (0.5,  'Qwen2.5', 'Q2.5-0.5B',  24, 14, 896),
    'Qwen2.5-1.5B-Instruct': (1.5,  'Qwen2.5', 'Q2.5-1.5B',  28, 12, 1536),
    'Qwen2.5-3B':            (3.0,  'Qwen2.5', 'Q2.5-3B-base',36, 16, 2048),
    'Qwen2.5-3B-Instruct':   (3.0,  'Qwen2.5', 'Q2.5-3B-inst',36, 16, 2048),
    'Qwen3.5-0.8B':          (0.8,  'Qwen3.5', 'Q3.5-0.8B',  28, 16, 1024),
    'Qwen3.5-2B':            (2.0,  'Qwen3.5', 'Q3.5-2B',    28, 16, 1536),
    'Qwen3.5-4B':            (4.0,  'Qwen3.5', 'Q3.5-4B',    36, 32, 2560),
    'Qwen3.5-9B':            (9.0,  'Qwen3.5', 'Q3.5-9B',    36, 32, 4096),
    'gemma-3-270m-it':       (0.27, 'Gemma3',  'Gemma-270M', 18,  4, 1152),
    'gemma-3-1b-it':         (1.0,  'Gemma3',  'Gemma-1B',   26,  4, 1152),
    'gemma-3-4b-it':         (4.0,  'Gemma3',  'Gemma-4B',   34,  8, 2560),
}
FAMILY_COLORS = {'Qwen2.5': '#e74c3c', 'Qwen3.5': '#3498db', 'Gemma3': '#27ae60'}

# ── load data ────────────────────────────────────────────────────────────────
df = pd.read_csv('../v3/results_vllm/local_models_behavioral.csv')
df['gap'] = df['fvq_acc'] - df['cvq_acc']
df['reversed'] = (df['cvq_acc'] > df['fvq_acc']).astype(int)
df['kn'] = df['num_keys'] * df['num_updates']

# add metadata
df['params']  = df['model'].map({k: v[0] for k,v in MODELS.items()})
df['family']  = df['model'].map({k: v[1] for k,v in MODELS.items()})
df['label']   = df['model'].map({k: v[2] for k,v in MODELS.items()})
df['n_heads'] = df['model'].map({k: v[4] for k,v in MODELS.items()})
df['d_model'] = df['model'].map({k: v[5] for k,v in MODELS.items()})

fam_df = df[df['family'].notna()].copy()

print(f"Rows: {len(fam_df)} | Models: {fam_df.model.nunique()}")
print()
print(fam_df.groupby('model')[['fvq_acc','cvq_acc','gap','reversed']]
      .agg({'fvq_acc':'mean','cvq_acc':'mean','gap':'mean','reversed':'sum'})
      .round(3).to_string())

## 1. Family Portrait — K×N Heatmaps (all 11 models)

Each cell: gap = FVQ − CVQ. Blue = reversal. Red = FVQ dominates.

In [ ]:
families = [
    ('Qwen2.5', ['Qwen2.5-0.5B-Instruct','Qwen2.5-1.5B-Instruct',
                  'Qwen2.5-3B','Qwen2.5-3B-Instruct']),
    ('Qwen3.5', ['Qwen3.5-0.8B','Qwen3.5-2B','Qwen3.5-4B','Qwen3.5-9B']),
    ('Gemma3',  ['gemma-3-270m-it','gemma-3-1b-it','gemma-3-4b-it']),
]

fig = plt.figure(figsize=(20, 14))
gs = GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.35)

for row_idx, (fam_name, models) in enumerate(families):
    fam_color = FAMILY_COLORS[fam_name]
    for col_idx, model in enumerate(models):
        ax = fig.add_subplot(gs[row_idx, col_idx])
        mdf = fam_df[fam_df.model == model]
        if len(mdf) == 0:
            ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
            continue

        pivot = mdf.pivot_table(index='num_updates', columns='num_keys',
                                values='gap', aggfunc='mean')
        im = ax.pcolormesh(pivot.columns, pivot.index, pivot.values,
                           cmap='RdBu_r', vmin=-0.7, vmax=0.7, shading='auto')

        # mark reversal cells
        for k in pivot.columns:
            for n in pivot.index:
                try:
                    v = pivot.loc[n, k]
                    if not np.isnan(v) and v < 0:
                        ax.scatter(k, n, s=25, c='black', marker='x', lw=1.2, zorder=5)
                except: pass

        info = MODELS.get(model, (0,'','',0,0,0))
        params, _, short, n_layers, n_heads, d_model = info
        n_rev = (mdf.gap < 0).sum()
        rev_rate = n_rev / len(mdf)

        title_color = '#8e44ad' if n_rev > 0 else '#2c3e50'
        ax.set_title(f"{short}\n{params}B | {n_rev} rev ({rev_rate:.0%})",
                     fontsize=8.5, color=title_color, fontweight='bold')
        ax.set_xlabel('K (keys)', fontsize=7)
        ax.set_ylabel('N (updates)', fontsize=7)
        ax.tick_params(labelsize=7)
        plt.colorbar(im, ax=ax, shrink=0.75, pad=0.02)

    # family label
    fig.text(0.01, 0.83 - row_idx * 0.32, fam_name,
             fontsize=13, fontweight='bold', color=fam_color,
             rotation=90, va='center')

fig.suptitle('Family Portrait: Gap = FVQ − CVQ in K×N Space\n'
             'Blue = CVQ > FVQ (reversal, ×) | Red = FVQ dominates',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('../v3/plots/family_heatmaps.png', bbox_inches='tight', dpi=150)
plt.show()

## 2. Reversal Boundary — min K causing reversal vs N

For each reversal-prone model: at each N level, what is the minimum K where reversal first appears?  
This traces the **capacity boundary curve** — below it, FVQ holds; above, CVQ wins.

In [ ]:
reversal_models = [
    ('Qwen2.5-3B',         'Q2.5-3B-base', '#c0392b', '--'),
    ('Qwen2.5-3B-Instruct','Q2.5-3B-inst', '#e74c3c', '-'),
    ('Qwen3.5-2B',         'Q3.5-2B',      '#2980b9', '--'),
    ('Qwen3.5-4B',         'Q3.5-4B',      '#3498db', '-'),
    ('Qwen2.5-1.5B-Instruct','Q2.5-1.5B',  '#e67e22', ':'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel A: min K causing reversal vs N
ax = axes[0]
for model, label, color, ls in reversal_models:
    mdf = fam_df[fam_df.model == model]
    if len(mdf) == 0: continue
    boundary = []
    for n in sorted(mdf.num_updates.unique()):
        ndf = mdf[mdf.num_updates == n]
        rev_k = ndf[ndf.gap < 0]['num_keys'].min()
        if not np.isnan(rev_k):
            boundary.append((n, rev_k))
    if boundary:
        ns, ks = zip(*boundary)
        ax.plot(ns, ks, marker='o', color=color, ls=ls, lw=2, ms=7, label=label)

ax.set_xlabel('N (num_updates)', fontsize=11)
ax.set_ylabel('Min K causing reversal', fontsize=11)
ax.set_title('A. Reversal boundary: min K vs N\n'
             'Lower curve = reverses more easily', fontsize=10)
ax.legend(fontsize=9)
ax.set_ylim(0, 35)
ax.fill_between([0, 110], [0,0], [35,35], alpha=0.04, color='blue')
ax.text(5, 2, 'Reversal zone (below curve)', fontsize=8, color='navy', alpha=0.7)

# Panel B: min K×N causing reversal vs params
ax = axes[1]
threshold_data = []
for model in MODELS:
    mdf = fam_df[fam_df.model == model]
    if len(mdf) == 0: continue
    rev_cells = mdf[mdf.gap < 0]
    info = MODELS[model]
    params, fam, short = info[0], info[1], info[2]
    if len(rev_cells) == 0:
        threshold_data.append({'model': model, 'label': short, 'params': params,
                                'family': fam, 'min_kn': np.nan, 'min_k': np.nan,
                                'n_rev': 0, 'rev_rate': 0})
    else:
        threshold_data.append({'model': model, 'label': short, 'params': params,
                                'family': fam,
                                'min_kn': rev_cells['kn'].min(),
                                'min_k':  rev_cells['num_keys'].min(),
                                'n_rev': len(rev_cells),
                                'rev_rate': len(rev_cells)/len(mdf)})

t_df = pd.DataFrame(threshold_data)

for fam, fam_color in FAMILY_COLORS.items():
    sub = t_df[t_df.family == fam]
    # Models WITH reversal
    rev = sub.dropna(subset=['min_kn'])
    no_rev = sub[sub.min_kn.isna()]

    if len(rev) > 0:
        ax.scatter(rev['params'], rev['min_kn'], s=120, c=fam_color,
                   zorder=4, label=f'{fam} (reverses)', edgecolors='white', lw=1)
        for _, row in rev.iterrows():
            ax.annotate(row['label'], (row['params'], row['min_kn']),
                       xytext=(5, 4), textcoords='offset points', fontsize=8,
                       color=fam_color)
    if len(no_rev) > 0:
        # Plot as lower-bound triangles at max observed KN
        max_kn = fam_df[fam_df.family == fam]['kn'].max()
        for _, row in no_rev.iterrows():
            ax.scatter(row['params'], max_kn, s=100, c=fam_color, marker='^',
                      alpha=0.5, zorder=3)
            ax.annotate(f"{row['label']}\n≥{int(max_kn)}",
                       (row['params'], max_kn),
                       xytext=(5, 2), textcoords='offset points',
                       fontsize=7, color=fam_color, alpha=0.7)

ax.set_xlabel('Model size (B params)', fontsize=11)
ax.set_ylabel('Min K×N causing reversal (lower = easier to reverse)', fontsize=10)
ax.set_title('B. Reversal threshold T_model vs params\n'
             'Circles = threshold | Triangles ▲ = never reversed (lower bound)',
             fontsize=10)

# Add legend patches
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=patches, fontsize=9)

plt.suptitle('Reversal Boundary Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../v3/plots/family_boundary.png', bbox_inches='tight', dpi=150)
plt.show()

print("\n=== Reversal threshold table ===")
print(t_df[['label','family','params','min_kn','min_k','n_rev','rev_rate']]
      .sort_values('params').round(2).to_string(index=False))

## 3. FVQ and CVQ Means vs Model Size (per family)

What changes as we go up the size ladder within each family?  
**Hypothesis:** Reversal happens where FVQ is in a 'medium' capability range — capable enough that CVQ via recency can catch it, but not stable enough to resist K-load.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

for ax, (fam_name, models) in zip(axes, families):
    fam_color = FAMILY_COLORS[fam_name]

    xs, fvqs, cvqs, rev_rates, labels = [], [], [], [], []
    for model in models:
        mdf = fam_df[fam_df.model == model]
        if len(mdf) == 0: continue
        p = MODELS[model][0]
        xs.append(p)
        fvqs.append(mdf.fvq_acc.mean())
        cvqs.append(mdf.cvq_acc.mean())
        rev_rates.append(mdf.reversed.mean())
        labels.append(MODELS[model][2])

    ax.plot(xs, fvqs, 'o-', color=fam_color, lw=2.5, ms=10,
            markeredgecolor='white', markeredgewidth=1.5, label='FVQ (mean)', zorder=4)
    ax.plot(xs, cvqs, 's--', color=fam_color, lw=2, ms=8, alpha=0.7,
            markeredgecolor='white', label='CVQ (mean)', zorder=3)

    # Shade reversal rate as bar width
    for x, rev in zip(xs, rev_rates):
        ax.axvspan(x - 0.02, x + 0.02, alpha=rev * 0.6,
                   color='#9b59b6', zorder=1)

    # Annotate reversal rate
    for x, rr, lbl in zip(xs, rev_rates, labels):
        ax.text(x, 0.02, f'{rr:.0%}\nrev', ha='center', va='bottom',
                fontsize=7.5, color='#8e44ad', fontweight='bold')
        ax.text(x, 1.03, lbl, ha='center', va='bottom',
                fontsize=7, color='gray', rotation=0)

    ax.axhline(0.5, color='gray', lw=0.8, ls=':', alpha=0.5)
    ax.fill_between(xs, fvqs, cvqs,
                    where=[c > f for f, c in zip(fvqs, cvqs)],
                    alpha=0.15, color='blue', label='CVQ > FVQ zone')

    ax.set_xlabel('Params (B)', fontsize=10)
    ax.set_ylabel('Accuracy (mean over all cells)', fontsize=10)
    ax.set_title(f'{fam_name}\nSolid=FVQ | Dashed=CVQ | Purple bar=reversal rate',
                 fontsize=10, fontweight='bold', color=fam_color)
    ax.set_ylim(-0.02, 1.1)
    ax.legend(fontsize=8, loc='upper left')

plt.suptitle('FVQ and CVQ Mean Accuracy vs Model Size by Family',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../v3/plots/family_fvq_cvq_vs_size.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. The Capacity Window Hypothesis

Prediction: reversal occurs when FVQ ∈ [0.3, 0.8] — capable enough to retrieve v_first in easy cells, but fragile under K-load. Below 0.3, both fail (no systematic reversal). Above 0.8, FVQ is too strong to fall below CVQ.  

Test: plot reversal rate vs FVQ level per cell across all family models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (fam_name, models) in zip(axes, families):
    mdf_fam = fam_df[fam_df.family == fam_name].copy()
    fam_color = FAMILY_COLORS[fam_name]

    # Bin by FVQ level
    mdf_fam['fvq_bin'] = pd.cut(mdf_fam['fvq_acc'], bins=np.arange(0, 1.05, 0.1))
    bin_stats = mdf_fam.groupby('fvq_bin', observed=True).agg(
        rev_rate=('reversed', 'mean'),
        n=('reversed', 'count'),
        fvq_mid=('fvq_acc', 'mean')
    ).dropna()

    ax.bar(range(len(bin_stats)), bin_stats['rev_rate'],
           color=fam_color, alpha=0.75, edgecolor='white')
    for i, (_, row) in enumerate(bin_stats.iterrows()):
        ax.text(i, row['rev_rate'] + 0.01, f'n={int(row["n"])}',
                ha='center', fontsize=7)

    ax.set_xticks(range(len(bin_stats)))
    ax.set_xticklabels([f'{b.left:.1f}–{b.right:.1f}'
                        for b in bin_stats.index], rotation=45, fontsize=7)
    ax.set_xlabel('FVQ accuracy bin', fontsize=10)
    ax.set_ylabel('Reversal rate', fontsize=10)
    ax.set_title(f'{fam_name}\nReversal rate by FVQ level', fontsize=10,
                 fontweight='bold', color=fam_color)
    ax.set_ylim(0, 1)

plt.suptitle('The Capacity Window Hypothesis:\n'
             'Reversal peaks at mid-range FVQ — capable enough for CVQ to win, fragile under K-load',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../v3/plots/family_capacity_window.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Reversal Threshold vs Architecture Properties

All three families are dense — active params = total params.  
But architecture varies: n_heads, d_model, n_layers.  
Which property best predicts the reversal threshold?

In [ ]:
arch_rows = []
for model, (params, fam, label, n_layers, n_heads, d_model) in MODELS.items():
    mdf = fam_df[fam_df.model == model]
    if len(mdf) == 0: continue
    rev = mdf[mdf.gap < 0]
    n_rev = len(rev)
    rev_rate = n_rev / len(mdf)
    min_k = rev['num_keys'].min() if n_rev > 0 else np.nan
    min_kn = rev['kn'].min() if n_rev > 0 else np.nan
    mean_fvq = mdf.fvq_acc.mean()
    mean_cvq = mdf.cvq_acc.mean()

    arch_rows.append({
        'model': model, 'label': label, 'family': fam,
        'params': params, 'n_layers': n_layers,
        'n_heads': n_heads, 'd_model': d_model,
        'd_head': d_model // n_heads,   # attention head dimension
        'capacity_score': params * d_model / 1000,  # rough capacity proxy
        'n_rev': n_rev, 'rev_rate': rev_rate,
        'min_k': min_k, 'min_kn': min_kn,
        'mean_fvq': mean_fvq, 'mean_cvq': mean_cvq,
    })

arch_df = pd.DataFrame(arch_rows)
print(arch_df[['label','family','params','n_layers','n_heads','d_model','d_head',
               'rev_rate','min_k','min_kn','mean_fvq','mean_cvq']].round(3).to_string(index=False))

In [ ]:
# Correlate reversal rate and min_k with architecture properties
arch_features = ['params', 'n_layers', 'n_heads', 'd_model', 'd_head', 'capacity_score',
                 'mean_fvq', 'mean_cvq']

print("Spearman correlations with reversal rate and min_k_threshold")
print(f"{'Feature':>20}  r(rev_rate)   r(min_k)   r(min_kn)")
print("-" * 60)
for feat in arch_features:
    sub = arch_df.dropna(subset=[feat])
    r1, _ = spearmanr(sub[feat], sub['rev_rate'])
    sub2 = arch_df.dropna(subset=[feat, 'min_k'])
    r2 = spearmanr(sub2[feat], sub2['min_k'])[0] if len(sub2) >= 3 else np.nan
    sub3 = arch_df.dropna(subset=[feat, 'min_kn'])
    r3 = spearmanr(sub3[feat], sub3['min_kn'])[0] if len(sub3) >= 3 else np.nan
    print(f"{feat:>20}:  {r1:+.3f}       {r2:+.3f}      {r3:+.3f}")

print()
print("Note: r(min_k) positive = larger property → higher K threshold → LESS reversal-prone")
print("      r(rev_rate) negative = larger property → less likely to reverse")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

properties = [
    ('params', 'Params (B)'),
    ('n_heads', 'n_heads'),
    ('d_model', 'd_model'),
    ('d_head', 'd_head = d_model/n_heads'),
]

for col, (prop, prop_label) in enumerate(properties):
    # Row 0: reversal rate vs property
    ax = axes[0, col]
    for fam, fam_color in FAMILY_COLORS.items():
        sub = arch_df[arch_df.family == fam]
        ax.scatter(sub[prop], sub['rev_rate'], c=fam_color, s=80,
                   label=fam, zorder=3, edgecolors='white')
        for _, row in sub.iterrows():
            ax.annotate(row['label'].split('-')[-1], (row[prop], row['rev_rate']),
                       xytext=(3, 2), textcoords='offset points', fontsize=7)
    r, p = spearmanr(arch_df[prop], arch_df['rev_rate'])
    ax.set_title(f'Reversal rate vs {prop_label}\nr={r:.3f} (p={p:.3f})', fontsize=9)
    ax.set_xlabel(prop_label, fontsize=9)
    ax.set_ylabel('Reversal rate' if col == 0 else '', fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    if col == 0:
        ax.legend(fontsize=8)

    # Row 1: min K threshold vs property (only models that reverse)
    ax = axes[1, col]
    sub_rev = arch_df.dropna(subset=['min_k'])
    for fam, fam_color in FAMILY_COLORS.items():
        sub = sub_rev[sub_rev.family == fam]
        ax.scatter(sub[prop], sub['min_k'], c=fam_color, s=80,
                   label=fam, zorder=3, edgecolors='white')
        for _, row in sub.iterrows():
            ax.annotate(row['label'].split('-')[-1], (row[prop], row['min_k']),
                       xytext=(3, 1), textcoords='offset points', fontsize=7)
    if len(sub_rev) >= 3:
        r, p = spearmanr(sub_rev[prop], sub_rev['min_k'])
        ax.set_title(f'Min K threshold vs {prop_label}\nr={r:.3f} (p={p:.3f})', fontsize=9)
    ax.set_xlabel(prop_label, fontsize=9)
    ax.set_ylabel('Min K causing reversal' if col == 0 else '', fontsize=9)

plt.suptitle('Architecture Properties vs Reversal Behavior\n'
             'Top row: reversal rate | Bottom row: min K threshold (models that reverse only)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../v3/plots/family_arch_correlations.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. The Base vs Instruct Effect (Qwen2.5-3B)

Same architecture, different training. Base reverses at K=7/N=7; Instruct at K=25/N=10.  
Instruction tuning raises the reversal threshold — why?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

base = fam_df[fam_df.model == 'Qwen2.5-3B']
inst = fam_df[fam_df.model == 'Qwen2.5-3B-Instruct']

# Panel A: FVQ and CVQ vs K×N
ax = axes[0]
for mdf, label, color, ls in [(base, 'Base', '#c0392b', '--'), (inst, 'Instruct', '#e74c3c', '-')]:
    mdf2 = mdf.copy()
    mdf2['kn_bin'] = pd.cut(np.log(mdf2['kn']+1), bins=10)
    g = mdf2.groupby('kn_bin', observed=True).agg(
        fvq=('fvq_acc','mean'), cvq=('cvq_acc','mean'), kn=('kn','mean')).dropna()
    ax.plot(g.kn, g.fvq, color=color, ls=ls, lw=2.5, marker='o', ms=6,
            label=f'{label} FVQ')
    ax.plot(g.kn, g.cvq, color=color, ls=':', lw=2, marker='s', ms=5,
            alpha=0.7, label=f'{label} CVQ')
ax.axhline(0.5, color='gray', lw=0.8, ls=':')
ax.set_xscale('log')
ax.set_xlabel('K×N (log)')
ax.set_ylabel('Accuracy')
ax.set_title('Qwen2.5-3B: Base vs Instruct\nFVQ/CVQ vs K×N')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)

# Panel B: Reversal rate as function of K (at fixed N=30)
ax = axes[1]
for mdf, label, color in [(base, 'Base', '#c0392b'), (inst, 'Instruct', '#e74c3c')]:
    n30 = mdf[mdf.num_updates.isin([20, 30])].copy()
    k_rev = n30.groupby('num_keys')[['gap','reversed']].mean().reset_index()
    ax.plot(k_rev['num_keys'], k_rev['reversed'], 'o-', color=color, lw=2, ms=7, label=label)
ax.axhline(0.5, color='gray', lw=0.8, ls=':', label='50% reversal')
ax.set_xlabel('K (num_keys)')
ax.set_ylabel('Reversal rate')
ax.set_title('Base vs Instruct: Reversal rate vs K\n(at N=20/30)')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)

# Panel C: Gap distribution comparison
ax = axes[2]
ax.hist(base['gap'], bins=25, alpha=0.6, color='#c0392b', label='Base', density=True)
ax.hist(inst['gap'], bins=25, alpha=0.6, color='#e74c3c', label='Instruct', density=True)
ax.axvline(0, color='k', lw=1, ls='--')
base_neg = (base.gap < 0).mean()
inst_neg = (inst.gap < 0).mean()
ax.text(0.05, 0.85, f'Base: {base_neg:.0%} reversed', transform=ax.transAxes,
        fontsize=9, color='#c0392b')
ax.text(0.05, 0.75, f'Inst:  {inst_neg:.0%} reversed', transform=ax.transAxes,
        fontsize=9, color='#e74c3c')
ax.set_xlabel('Gap (FVQ − CVQ)')
ax.set_ylabel('Density')
ax.set_title('Gap distribution: Base vs Instruct')
ax.legend(fontsize=9)

plt.suptitle('Base vs Instruct: Same Architecture, Different Reversal Threshold',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../v3/plots/base_vs_instruct.png', bbox_inches='tight', dpi=150)
plt.show()

print("Base:    FVQ={:.3f}  CVQ={:.3f}  gap={:.3f}".format(
    base.fvq_acc.mean(), base.cvq_acc.mean(), base.gap.mean()))
print("Instruct: FVQ={:.3f}  CVQ={:.3f}  gap={:.3f}".format(
    inst.fvq_acc.mean(), inst.cvq_acc.mean(), inst.gap.mean()))
print("\nInstruct tuning suppresses CVQ recency signal AND improves FVQ stability → higher threshold")

## 7. Gemma-3: Why No Reversal at 1B and 4B?

Gemma-3 is an outlier: 270M barely reverses (floor noise), 1B and 4B never reverse.  
CVQ at 1B is basically 0.045 — the model produces zero CVQ accuracy. It's architecturally incapable of recency-based retrieval.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

gemma_models = ['gemma-3-270m-it', 'gemma-3-1b-it', 'gemma-3-4b-it']
gemma_labels = ['Gemma-270M', 'Gemma-1B', 'Gemma-4B']
gemma_colors = ['#a8d5a2', '#4CAF50', '#1B5E20']

for ax, model, label, color in zip(axes, gemma_models, gemma_labels, gemma_colors):
    mdf = fam_df[fam_df.model == model]
    if len(mdf) == 0:
        ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
        continue

    # FVQ and CVQ vs K×N
    mdf2 = mdf.copy()
    mdf2['kn_bin'] = pd.cut(np.log(mdf2['kn']+1), bins=8)
    g = mdf2.groupby('kn_bin', observed=True).agg(
        fvq=('fvq_acc','mean'), cvq=('cvq_acc','mean'), kn=('kn','mean')).dropna()

    ax.scatter(mdf['kn'], mdf['fvq_acc'], alpha=0.2, s=12, c=color, label='FVQ (raw)')
    ax.scatter(mdf['kn'], mdf['cvq_acc'], alpha=0.2, s=12, c='gray', label='CVQ (raw)')
    if len(g) > 1:
        ax.plot(g.kn, g.fvq, '-o', color=color, lw=2.5, ms=7, label='FVQ (binned mean)')
        ax.plot(g.kn, g.cvq, '--s', color='gray', lw=2, ms=6, label='CVQ (binned mean)')

    ax.axhline(0.5, color='gray', lw=0.7, ls=':', alpha=0.6)
    ax.set_xscale('log')
    ax.set_xlabel('K×N (log scale)')
    ax.set_ylabel('Accuracy')
    fvq_m = mdf.fvq_acc.mean()
    cvq_m = mdf.cvq_acc.mean()
    n_rev = (mdf.gap < 0).sum()
    ax.set_title(f'{label}\nFVQ={fvq_m:.3f}  CVQ={cvq_m:.3f}  rev={n_rev}',
                 fontsize=10, fontweight='bold', color=color)
    ax.set_ylim(-0.02, 1.05)
    ax.legend(fontsize=8)

plt.suptitle('Gemma-3 Family: CVQ Stays Near Zero — Recency Retrieval Fails\n'
             'No reversal because CVQ never gets close enough to FVQ to overtake',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../v3/plots/gemma3_cvq_profile.png', bbox_inches='tight', dpi=150)
plt.show()

print("CVQ profile for Gemma-3:")
for model, label in zip(gemma_models, gemma_labels):
    mdf = fam_df[fam_df.model == model]
    print(f"  {label:12s}: mean CVQ={mdf.cvq_acc.mean():.3f}  "
          f"max CVQ={mdf.cvq_acc.max():.3f}  "
          f"cells with CVQ>0.3: {(mdf.cvq_acc > 0.3).sum()}/{len(mdf)}")

## 8. Summary: What Determines the Reversal Threshold?

Putting it all together.

In [ ]:
print("=" * 72)
print("FAMILY REVERSAL ANALYSIS — SUMMARY")
print("=" * 72)

print("\n--- WHICH MODELS REVERSE ---")
for model, (params, fam, label, n_layers, n_heads, d_model) in MODELS.items():
    mdf = fam_df[fam_df.model == model]
    if len(mdf) == 0: continue
    rev = mdf[mdf.gap < 0]
    if len(rev) == 0:
        print(f"  {label:18s} ({params}B): NO REVERSAL | "
              f"FVQ={mdf.fvq_acc.mean():.2f} CVQ={mdf.cvq_acc.mean():.2f}")
    else:
        first = rev.nsmallest(1, 'kn').iloc[0]
        print(f"  {label:18s} ({params}B): REVERSAL {len(rev):3d} cells | "
              f"first at K={int(first.num_keys)}/N={int(first.num_updates)} "
              f"(KN={int(first.kn)}) | "
              f"FVQ={mdf.fvq_acc.mean():.2f} CVQ={mdf.cvq_acc.mean():.2f}")

print()
print("--- WHAT DETERMINES REVERSAL THRESHOLD ---")
print()
print("1. MEAN FVQ level is the strongest architectural predictor:")
r, p = spearmanr(arch_df['mean_fvq'], arch_df['rev_rate'])
print(f"   r(mean_fvq, rev_rate) = {r:.3f} (p={p:.4f})")
print("   High FVQ → strong primacy → harder to reverse")
print()
print("2. n_heads matters — more heads = more capacity for QK routing to v_first:")
r, p = spearmanr(arch_df['n_heads'], arch_df['rev_rate'])
print(f"   r(n_heads, rev_rate) = {r:.3f} (p={p:.4f})")
print()
print("3. Base vs Instruct: same architecture, instruct tuning raises threshold")
print("   Qwen2.5-3B-base:    first reversal at KN=49  (K=7/N=7)")
print("   Qwen2.5-3B-instruct: first reversal at KN=150 (K=25/N=10)")
print("   → SFT suppresses recency bias, reinforces primacy binding")
print()
print("4. Gemma-3 never reverses because CVQ ≈ 0 — recency retrieval is structurally absent")
print("   CVQ needs the model to resolve 'current value' via recency attention.")
print("   Gemma's 4-head GQA architecture may lack the head capacity for this.")
print()
print("5. Qwen3.5-2B anomaly: reversal at K=5/N=5 (KN=25) — earliest of any model")
print("   But reversal DECREASES with more N. K is the sole driver for this model.")
print("   Qwen3.5-9B: zero reversals. The 2B→9B jump eliminates reversal entirely.")
print()
print("--- THE CAPACITY WINDOW ---")
print("Reversal requires TWO conditions to hold simultaneously:")
print("  A) FVQ breaks under K-load  (model can't maintain first binding at high K)")
print("  B) CVQ is viable            (model can retrieve v_last via recency)")
print("Models too small: both fail (Gemma-270M noise, Qwen2.5-0.5B)")
print("Models in the window: FVQ fragile, CVQ viable → reversal")
print("Models past the window: FVQ too strong to break → no reversal")
print("Gemma-3: violates condition B — CVQ structurally absent → never reverses")